# Setup

In [0]:
%run ../config

In [0]:
%run ../helpers

## Imports

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from functools import reduce
from pyspark.sql import Window, DataFrame

from pyspark.sql.functions import col, max, lit, row_number, rank, min, year, current_timestamp, size, monotonically_increasing_id, round, count_distinct

## Variable

### Static Variable

In [0]:
SOURCE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.crimes"
TARGET_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.crimes"
CONTROL_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.load_control"
QUARANTINE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.crimes_quarantine"

In [0]:
# setup control schema for control table in silver layer

CONTROL_SCHEMA = StructType([
    StructField("table_name", StringType(), False),
    StructField("last_ingested_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
])

In [0]:
COLUMN_TYPES = {
    "id": "bigint",
    "case_number": "string",
    "date": "timestamp_ntz",
    "block": "string",
    "iucr": "string",
    "primary_type": "string",
    "description": "string",
    "location_description": "string",
    "arrest": "boolean",
    "domestic": "boolean",
    "beat": "string",
    "district": "string",
    "ward": "int",
    "community_area": "int",
    "fbi_code": "string",
    "x_coordinate": "double",
    "y_coordinate": "double",
    "year": "int",
    "updated_on": "timestamp_ntz",
    "latitude": "double",
    "longitude": "double",
}

DROPPED_COLUMNS = ["location"]

In [0]:
EXPECTED_BRONZE_COLUMNS = list(COLUMN_TYPES) + DROPPED_COLUMNS + ["_ingested_at"]

In [0]:
# safty limit for the quarantine
# If more than this % of the Bronze rows in a batch are quarantined, the run FAILS

MAX_QUARANTINE_PCT = 20.0

In [0]:
# one rule = one dictionary, made by make_check

ACTION_TAG = {"warn": "WARN", "null": "NULL", "quarantine": "QUARANTINE"}

### Dynamic Variable

In [0]:
quarantine_counts = {} # how many rows each topic quarantined 

In [0]:
# small helper for the `_reasons` column

UNFLAGGED = size(col("_reasons")) == 0

## Widget

In [0]:
dbutils.widgets.dropdown(
    name='run_mode',
    defaultValue='incremental',
    choices=['incremental', 'full_reprocess'],
    label='run_mode'
)

In [0]:
run_mode = dbutils.widgets.get('run_mode')

## Watermark

In [0]:
def get_last_watermark():

    # first run: the control table does not exist yet
    if not spark.catalog.tableExists(CONTROL_TABLE):
        print(f"[CONTROL TABLE] {CONTROL_TABLE} does not exist yet - first run, reading all of Bronze")
        return None
    
    # take the newest watermark saved for this table
    control_rows = spark.table(CONTROL_TABLE).filter(col("table_name") == 'crimes')
    last_watermark = control_rows.agg(max("last_ingested_at")).collect()[0][0]

    # control table exists but has no row for crimes yet
    if last_watermark is None:
        print(f"[CONTROL TABLE] no row for 'crimes' yet - first run, reading all of Bronze")

    return last_watermark

In [0]:
if run_mode == "full_reprocess":
    last_watermark = None
    print("[FULL REPROCESS] ignoring the watermark - reading all of Bronze")
else:
    last_watermark = get_last_watermark()
    print(f"[INCREMENTAL] reading Bronze rows with _ingested_at > {last_watermark}")

# Data Validation

In [0]:
bronze = spark.table(SOURCE_TABLE)

## TOPIC 1: Schema

a missing column or a wrong `_ingested_at` stops the run (FAIL) due to a missing column means the source changed; nothing after this point can be trusted. An extra column is harmless but should be noticed.

Steps: 
1. Set Up  
2. Profile 
3. ~~Quarantine~~ (none: a schema problem is about the whole batch, not one row)
4. ~~Transform~~ (none)
5. Verify



In [0]:
# 1. [SET UP] the expected columns are EXPECTED_BRONZE_COLUMNS 

print(f"[INFO]-[T1][SET UP]: Bronze must contain {len(EXPECTED_BRONZE_COLUMNS)} columns")

In [0]:
# 2. [PROFILE] compare the real Bronze columns with the expected ones

bronze_types = dict(bronze.dtypes)

missing_columns = [c for c in EXPECTED_BRONZE_COLUMNS if c not in bronze_types]
unexpected_columns = [c for c in bronze_types if c not in EXPECTED_BRONZE_COLUMNS]

# every data column must be text
not_text_columns = {
    c: t for c, t in bronze_types.items() if c in EXPECTED_BRONZE_COLUMNS and c != "_ingested_at" and t != "string"
}

print(f"[INFO]-[T1][PROFILE]: missing columns: {missing_columns}")
print(f"[INFO]-[T1][PROFILE]: unexpected columns: {unexpected_columns}")
print(f"[INFO]-[T1][PROFILE]: columns that are not text: {not_text_columns}")

In [0]:
# 5. [VERIFY] FAIL on a missing column or a wrong _ingested_at type

if missing_columns:
    raise ValueError(f"[ERROR]-[T1][VERIFY]: Bronze table {SOURCE_TABLE} is missing columns: {missing_columns}")

if not bronze_types["_ingested_at"].startswith("timestamp"):
    raise ValueError(f"[ERROR]-[T1][VERIFY]: _ingested_at must be a timestamp, found {bronze_types['_ingested_at']}")

if unexpected_columns:
    print(f"[WARN]-[T1][VERIFY]: columns not in COLUMN_TYPES will be ignored: {unexpected_columns}. Add them to COLUMN_TYPES if they are needed.")
if not_text_columns:
    print(f"[WARN]-[T1][VERIFY]: these Bronze columns are not text, so the text rules may behave differently: {not_text_columns}")
if not unexpected_columns and not not_text_columns:
    print("[OK]-[T1][VERIFY]: Bronze has exactly the expected columns")

### Read the batch

this step need to be process right after SCHEMA validation, because it needs `_ingested_at` and a valid schema

In [0]:
# find the newest Bronze row. This becomes the upper end of this batch.

batch_end = bronze.agg(max("_ingested_at")).collect()[0][0]

if batch_end is None:
    dbutils.notebook.exit("[EXIT]-[T1][READ]: Bronze Table is EMPTY!")

In [0]:
df_raw = bronze.filter(col("_ingested_at") <= lit(batch_end))

if last_watermark is not None:
    df_raw = df_raw.filter(col("_ingested_at") > lit(last_watermark))

raw_rows = df_raw.count()
print(f"[INFO]-[T1][READ]: ({last_watermark}, {batch_end}] -> {raw_rows:,} Bronze rows")

if raw_rows == 0:
    dbutils.notebook.exit("[EXIT]-[T1][READ]: no Bronze rows after the watermark")

In [0]:
# `_raw` is a copy of the original bronze row, kept untouched. The quarantined table is written from `_raw`, so a quarantined row always shows exactly what Bronze contained.

df = df_raw.select(
    *EXPECTED_BRONZE_COLUMNS,
    struct(*[col(c) for c in EXPECTED_BRONZE_COLUMNS]).alias("_raw"),
    array().cast("array<string>").alias("_reasons")
)

## TOPIC 2: Blanks and Text Normalisation

spaces around a value are trimmed; an empty text or a text such as `None`/`null`/`NaN` becomes `NULL`; the columns in `UPPERCASE_COLUMNS` are converted to upper case.

Step 
1. Set Up
2. Profile
3. ~~Quarantine~~(none: cleaning text never makes a row unusable)
4. Transform
5. Verify

In [0]:
# 1. [SET UP] which texts mean "missing", and which columns must be upper case

NULL_TOKENS = ["", "none", "null", "nan", "n/a"]

UPPERCASE_COLUMNS = ["case_number", "block", "iucr", "primary_type", "description", "location_description"]
UNTOUCHED_COLUMNS = ["fbi_code"]

STRING_COLUMNS = [c for c, t in df.dtypes if t == "string" and c not in UNTOUCHED_COLUMNS]

def normalize_text(name):
    trimmed = trim(col(name))
    return when(lower(trimmed).isin(*NULL_TOKENS), None).otherwise(trimmed)

In [0]:
# 2. [PROFILE] how many values will change? 
# change = the cleaned value (extra spaces, or a "missing" text)

changed_checks = [make_check(f"{c}_text_changed", [c], ~col(c).eqNullSafe(normalize_text(c)), "null") for c in STRING_COLUMNS]
changed = count_violations(df, changed_checks)

print(f"[INFO]-[T2][PROFILING]: values with extra spaces or a 'missing' text:")
for c in STRING_COLUMNS:
    n = changed[f"{c}_text_changed"]
    if n > 0:
        print(f"{c}: {n:,}")
        display(df.filter(~col(c).eqNullSafe(normalize_text(c))).select(col(c)).distinct().limit(5))

# columns that must be upper case: how many values are not?
case_checks = [make_check(f"{c}_not_uppercase", [c], col(c).isNotNull() & (col(c) != upper(col(c))), "null") for c in UPPERCASE_COLUMNS]

not_upper = count_violations(df, case_checks)
for c in UPPERCASE_COLUMNS:
    n = not_upper[f"{c}_not_uppercase"]
    print(f"[{'INFO' if n else 'OK'}]-[T2][PROFILING]: {c}: {n:,} values are not upper case" + (" -> will be converted" if n else ""))

In [0]:
# 4. [TRANSFORM] trim + "missing" texts -> NULL for every text column, then upper case for UPPERCASE_COLUMNS

df = df.select([normalize_text(c).alias(c) if c in STRING_COLUMNS else col(c) for c in df.columns])

for c in UPPERCASE_COLUMNS:
    df = df.withColumn(c, upper(col(c)))

In [0]:
# 5. [VERIFY] nothing left to clean 
verify_checks(df, changed_checks + case_checks, "T2")

## TOPIC 3: Data Types

a value that cannot be read as the type in `COLUMN_TYPES` is marked for quarantine (reason `<column>_unreadable`)

Step 
1. Set Up
2. Profile
3. Quarantine
4. Transform
5. Verify

In [0]:
# 1. [SET UP] how to read each column as its type

def cast_value(name, type_name):
    column = col(name)

    if type_name == "string":
        return column
    if type_name == "boolean":
        return when(lower(column).isin("true", "false"), lower(column) == "true")
    return column.try_cast(type_name)
        
TYPED_COLUMNS = {c: t for c, t in COLUMN_TYPES.items() if t != "string"}

In [0]:
# 2. [PROFILE] values that are present but cannot be read as their type

cast_checks = [make_check(f"{c}_unreadable", [c], col(c).isNotNull() & cast_value(c, t).isNull(), "quarantine") for c, t in TYPED_COLUMNS.items()]

show_violations(df, cast_checks, "T3")

In [0]:
# 3. [QUARANTINE] mark the rows with an unreadable value 

df = mark_quarantine(df, cast_checks, "type")

In [0]:
# 4. [TRANSFORM] cast every column to its final type

dropped_now = [c for c in df.columns if c not in COLUMN_TYPES and c not in ("_raw", "_ingested_at", "_reasons")]

print(f"[INFO]-[T3][TRANSFORM]: dropped columns: {dropped_now}")

df_before_cast = df

df = df.select(
    *[cast_value(c, t).alias(c) for c, t in COLUMN_TYPES.items()],
    "_raw",
    "_reasons",
    "_ingested_at",
)

In [0]:
# 5. [VERIFY] (1) every unreadable value is marked, (2) the real types are the declared types

verify_checks(df_before_cast, [c for c in cast_checks if c["action"] == "quarantine"], "T3")

real_types = dict(df.dtypes)
wrong_types = {c: (t, real_types[c]) for c, t in COLUMN_TYPES.items() if real_types[c] != t}

if wrong_types:
    raise ValueError(f"[ERROR]-[T3][VERIFY]: column types differ from COLUMN_TYPES (expected, real): {wrong_types}")

print("[OK]-[T3][VERIFY]: every column has its declared type")
df.printSchema()

## TOPIC 4: Key and Duplicate

a row without a usable key (`id`) or without `updated_on` is marked for quarantine. For an `id` that appears several times, only the newest version is kept (`updated_on`, then `_ingested_at` for a tie)

**Details of the Dedupe**:
1. A row without a key cannot be grouped with its versions. stays visible and is quarantined.
2. If an `id` has several rows and one of them has no usable `updated_on`, I cannot tell which row is the newest, so I do not choose. Every row of that `id` is marked `ambiguous_version` and go to the quarantine table
3. Rows that topic 3 marked take part in the dedupe too. If the newest version of an `id` is bad, the older versions are dropped and the bad one is quarantined, so an outdated version can never slip into Silver.

Step 
1. Set Up
2. Profile
3. Quarantine
4. Transform (dedupe)
5. Verify

In [0]:
# 1. [SET UP] which column identify a row, and which one says "newest"

KEY_COLUMNS = ["id"]
VERSION_COLUMN = "updated_on"

has_key_columns = reduce(lambda a, b: a & b, [col(c).isNotNull() for c in KEY_COLUMNS])

def add_version_groups(df):
    # _group = the key as text. A row without key gets a group of its own (os it is never grouped with other rows).
    # _ambiguous = True for every row of a key that has several rows in this batch AND at least one row without a usable.

    df = df.withColumn("_group", when(has_key_columns, concat_ws("|", *[col(c).cast("string") for c in KEY_COLUMNS])).otherwise(concat(lit("no-key-"), monotonically_increasing_id().cast("string"))))

    same_key = Window.partitionBy("_group")
    
    return df.withColumn("_ambiguous", (count(lit(1)).over(same_key) > 1) & (max(col(VERSION_COLUMN).isNull().cast("int")).over(same_key) == 1))

KEY_CHECKS = (
    [make_check(f"{k}_invalid", [k], is_missing(k), "quarantine") for k in KEY_COLUMNS] +
    [make_check(f"{VERSION_COLUMN}_invalid", [VERSION_COLUMN], is_missing(VERSION_COLUMN), "quarantine")] +
    [make_check("ambiguous_version", KEY_COLUMNS + [VERSION_COLUMN], col("_ambiguous"), "quarantine")] 
)

In [0]:
# 2. [PROFILE] rows without a usable key, keye whose newest version cannot be decided, and how many versions each key has

show_violations(add_version_groups(df), KEY_CHECKS, "T4")

has_key = reduce(lambda a, b: a & b, [col(c).isNotNull() for c in KEY_COLUMNS + [VERSION_COLUMN]])
versions_per_key = df.filter(has_key).groupBy(*KEY_COLUMNS).count()

distinct_keys = versions_per_key.count()
keys_with_many_versions = versions_per_key.filter(col("count") > 1).count()
max_versions = versions_per_key.agg(max("count")).collect()[0][0]

print(f"[INFO]-[T4][PROFILE]: {distinct_keys:,} distinct keys in this batch; {keys_with_many_versions:,} have more than one version (max {max_versions})")

In [0]:
# 3. [QUARANTINE] mark the rows without a usable key or updated_on, and every row of a key whose newest version cannot be decided 

df = mark_quarantine(add_version_groups(df), KEY_CHECKS, "key").drop("_group", "_ambiguous")

In [0]:
# 4. [TRANSFORM] number the versions of each key from newest to oldest
# rn = 1,2,3,... always uniuqe inside a group
# rk = like rn, but rows that tie get the same number (used only to detect ties)
# a key whose newest version cannot be decided (_ambigious) skips the dedupe (_ambiguous) skips the dedupe

rows_before_dedupe = df.count()

df = add_version_groups(df)
latest_window = (
    Window.partitionBy("_group")
        .orderBy(
            col(VERSION_COLUMN).desc(), 
            col("_ingested_at").desc()
        )
)

ranked = df.withColumn("rn", row_number().over(latest_window)).withColumn("rk", rank().over(latest_window))

# WARN: a row with rn > 1 but rk = 1 with the chosen row on BOTH updated_on and _ingested_at, so which one is kept is arbitrary (harmless if the rows are identical, a problem if they differ).

tied_rows = ranked.filter((col("rn") > 1) & (col("rk") == 1) & ~col("_ambiguous"))
tied_count = tied_rows.count()

if tied_count > 0:
    print(f"[WARN]-[T4][TRANSFORM]: {tied_count:,} rows tie with the chosen version - the choice between them is arbitrary. Examples:")
    display(tied_rows.select(*KEY_COLUMNS, VERSION_COLUMN, "_ingested_at", "arrest", "domestic", "primary_type").limit(10))
else:
    print("[OK]-[T4][TRANSFORM]: no ties - the latest version of every key is unambiguous")
    
# keep only the newest version (and every row of an ambiguous key)
df = ranked.filter((col("rn") == 1) | col("_ambiguous")).drop("rn", "rk", "_group", "_ambiguous")


In [0]:
# 5. [VERIFY] every bad row is marked, and the key must be unique

verify_checks(add_version_groups(df), KEY_CHECKS, "T4")

latest_rows = df.count()
dropped_versions = rows_before_dedupe - latest_rows

duplicate_keys = (df.filter(has_key_columns & ~array_contains(col("_reasons"), "ambiguous_version")).groupBy(*KEY_COLUMNS).count().filter(col("count") > 1).count())

if duplicate_keys > 0:
    raise ValueError(f"[ERROR]-[T4][VERIFY]:  {KEY_COLUMNS} is not unique after the dedupe: {duplicate_keys:,} keys appear more than once")

print(f"[OK]-[T4][VERIFY]: {rows_before_dedupe:,} rows -> {latest_rows:,} rows (1 per key, except ambiguous keys); {dropped_versions:,} older versions dropped")

## TOPIC 5: Required columns and null rate

a row that has NULL in a required column is marked for quarantine. a dimension is built from each required column: 

1. `case_number`, `arreset` and `domestic` for `dim_arrest_status` (SCD Type 2)
2. `iucr` for `dim_crime_type`
3. `date` for `dim_date` and `dim_time`

A row without one of them cannot be placed in the star schema and replacing the missing value by NULL would not help.

Step
1. Set Up
2. Profile
3. Quarantine
4. ~~Transform~~
5. Verify

In [0]:
# 1. [SET UP] the required columns
REQUIRED_COLUMNS = ["case_number", "date", "iucr", "arrest", "domestic"]
REQUIRED_CHECKS = [make_check(f"{c}_invalid", [c], is_missing(c), "quarantine") for c in REQUIRED_COLUMNS]

In [0]:
# 2. [PROFILE] 

# (1) rows that miss a required column

show_violations(df, REQUIRED_CHECKS, "T5")

# (2) null rate of every column, over the rows that are not marked so far

data_columns = [c for c in df.columns if c not in ("_raw", "_reasons")]
null_pct = unflagged(df).agg(*[round(100 * avg(col(c).isNull().cast("int")), 3).alias(c) for c in data_columns]).collect()[0].asDict()

display(
    spark.createDataFrame([(c, None if null_pct[c] is None else float(null_pct[c])) for c in data_columns], "column string, null_pct double").orderBy(col("null_pct").desc())
)

In [0]:
# 3. [QUARANTINE] mark the rows that miss a required column

df = mark_quarantine(df, REQUIRED_CHECKS, "required")

In [0]:
# 5. [VERIFY] every row with a missing required column is marked 
verify_checks(df, REQUIRED_CHECKS, "T5")

## TOPIC 6: Ranges

per column: a value outside its range is marked for quarantine

Step 
1. Set Up
2. Profile
3. Quarantine
4. Transform
5. Verify

In [0]:
# 1. [SET UP] the allowed range of each column (None = no limit):

MIN_DATE = "2001-01-01"
MAX_YEAR = int(str(batch_end)[:4])

RANGE_RULES = {
    "date": (MIN_DATE, str(batch_end), "quarantine"),
    "year": (2001, MAX_YEAR, "quarantine"),
    "ward": (1, 50, "quarantine"),
    "community_area": (1, 77, "quarantine"),
    "latitude": (41.60, 42.10, "quarantine"), # a little wider than Chicago (about 41.64 to 42.02)
    "longitude": (-88.00, -87.50, "quarantine"), # (about -87.94 to -87.52)
    "x_coordinate": (1_100_000, 1_300_000, "quarantine"), # GUESS - There is no actual range about this column
    "y_coordinate": (1_800_000, 2_000_000, "quarantine") # GUESS - There is no actual range about this column
}

# numeric and date columns that need no range 
RANGE_NOT_NEEDED = ["id", "updated_on"] # updated_on only used to order version

def out_of_range(name, low, high):
    type_name = COLUMN_TYPES[name]
    bad = (
        (col(name) < lit(str(low)).cast(type_name)) | (col(name) > lit(str(high)).cast(type_name))
    )
    return coalesce(bad, lit(False))

RANGE_CHECKS = [make_check(f"{c}_out_of_range", [c], out_of_range(c, low, high), action) for c, (low, high, action) in RANGE_RULES.items()]

# a numeric/date column with no rule would pass
unchecked = [c for c, t in COLUMN_TYPES.items()
             if t in ("int", "bigint", "double", "timestamp_ntz") and c not in RANGE_RULES and c not in RANGE_NOT_NEEDED]

if unchecked:
    print(f"[WARN]-[T6][SET UP]: numeric/date columns without a range rule: {unchecked}")
else:
    print("[OK]-[T6][SET UP]: every numeric and date column has a range rule (or is listed in RANGE_NOT_NEEDED)")

In [0]:
# 2. [PROFILE]

# (1) smallesst and largest value really seen, next to the allowed range

observed = df.agg(*[e for c in RANGE_RULES for e in (min(c).alias(f"{c}_min"), max(c).alias(f"{c}_max"))]).collect()[0]

display(spark.createDataFrame(
    [(c, str(observed[f"{c}_min"]), str(observed[f"{c}_max"]), str(low), str(high), action)
     for c, (low, high, action) in RANGE_RULES.items()],
    "column string, observed_min string, observed_max string, allowed_min string, allowed_max string, action string",
))

# (2) how many coordinate values are exactly 0; these rows go to the quarantine table

coordinate_columns = ["x_coordinate", "y_coordinate", "latitude", "longitude"]
zero_counts = df.agg(*[sum((col(c) == 0).cast("int")).alias(c) for c in coordinate_columns]).collect()[0]

display(spark.createDataFrame([(c, int(zero_counts[c]or 0)) for c in coordinate_columns], "column string, rows_equal_to_0 long"))

# (3) rows outside the range

show_violations(df, RANGE_CHECKS, "T6")

In [0]:
# 3. [QUARANTINE] mark the rows outside the range

df = mark_quarantine(df, RANGE_CHECKS, "range")

In [0]:
# 4. [TRANSFORM] values outside the range become NULL (rules whose action is "null": none for now; a rule switched to "null" is handled here)

df = apply_nulling(df, RANGE_CHECKS, "T6")

In [0]:
# 5. [VERIFY] no value is outside its range any more, and every out-of-range is marked (FAIL otherwise)

verify_checks(df, RANGE_CHECKS, "T6")

## TOPIC 7: Formats

If data is not formatted expectedly, the whole row is marked and waits in the quarantine table

Step 
1. Set Up
2. Profile
3. Quarantine
4. Transform (only if a rule is set to `null`)
5. Verify

In [0]:
# 1. [SET UP] the format of each column, as a regular expression -> column: (pattern, action)

BLOCK_PATTERN = r"^([0-9]+X{0,2}) ([NSEW]) (.+)$" # <hundred, eg. 082XX> <direction N/S/E/W> <street name and type>

FORMAT_RULES = {
    "case_number": (r"^[A-Z]{2}[0-9]{6}$", "quarantine"), # 2 letters + 6 digits, e.g. JK123456
    "iucr": (r"^[0-9A-Z]{4}$", "quarantine"), # 4 characters, e.g. 0486 or 031A
    "beat": (r"^[0-9]+$", "quarantine"), # digits only
    "district": (r"^[0-9]+$", "quarantine"), # digits only
    "block": (BLOCK_PATTERN, "quarantine")
}

# True when the value exists but does not match the pattern 
FORMAT_CHECKS = [make_check(f"{c}_bad_format", [c], coalesce(col(c).isNotNull() & ~col(c).rlike(pattern), lit(False)), action) for c, (pattern, action) in FORMAT_RULES.items()]

In [0]:
# 2. [PROFILE] 

# (1) show values that do not match their format 

show_violations(df, FORMAT_CHECKS, "T7")

# (2) beat and district: how long are the values? (a code is usually always the same length)

for c in ["beat", "district"]:
    display(
        df.groupBy(length(c).alias(f"{c}_length"))
        .count()
        .orderBy(f"{c}_length")
    )

In [0]:
# 3. [QUARANTINE] mark the rows whose value does not match its format

df = mark_quarantine(df, FORMAT_CHECKS, "format")

In [0]:
# 4. [TRANSFORM] rules whose action is "null" (none for now)

df = apply_nulling(df, FORMAT_CHECKS, "T7")

In [0]:
# 5. [VERIFY] every row with a wrong format is marked (FAIL otherwise)

verify_checks(df, FORMAT_CHECKS, "T7")

## TOPIC 8: Allowed values and references

`quarantine` for a row whose value is not in the reference list is marked and waits in the quarantine table.

Step:
1. Set Up
2. Profile
3. Quarantine
4. ~~Transform~~
5. Verify

In [0]:
# 1. [SET UP]

REFERENCE_TABLES = {
    "community_area": (f"{CATALOG}.{SILVER_SCHEMA}.community_area_boundaries", "community_area_code", "quarantine"), # built by transform_boundaries
    "iucr": (f"{CATALOG}.{SILVER_SCHEMA}.ref_iucr", "iucr", "quarantine") # static copy of CPD dataset c7ck-438e
}

# load each reference that exists and turn it into a rule

REFERENCE_CHECKS = []
for column, (table, table_column, action) in REFERENCE_TABLES.items():
    if not spark.catalog.tableExists(table):
        print(f"[SKIP]-[T8][SET UP]: {column} - reference table {table} does not exist yet")
        continue

    ref_value = col(table_column).cast(COLUMN_TYPES[column])
    if COLUMN_TYPES[column] == "string":
        ref_value = upper(trim(ref_value))
    allowed = [r[0] for r in spark.table(table).select(ref_value).distinct().collect() if r[0] is not None]

    if not allowed:
        print(f"[SKIP]-[T8][SET UP]: reference table {table} is empty")
        continue

    print(f"[INFO]-[T8][SET UP]: {len(allowed):,} allowed values from {table}")

    # True when the value exists but is not in the reference list
    REFERENCE_CHECKS.append(
        make_check(
            f"{column}_not_in_reference", 
            [column], 
            coalesce(col(column).isNotNull() & ~col(column).isin(*allowed), 
            lit(False)), 
            action
        )
    )

In [0]:
# 2. [PROFILE] 

if REFERENCE_CHECKS:
    show_violations(df, REFERENCE_CHECKS, "T8")
else:
    print("[SKIP]-[T8][PROFILE]: no reference table exists yet - nothing to check")

In [0]:
# 3. [QUARANTINE] mark the rows whose value is not in its reference list 

df = mark_quarantine(df, REFERENCE_CHECKS, "reference")

In [0]:
# 5. [VERIFY] every row with a value outside its reference is marked (FAIL otherwise)

if REFERENCE_CHECKS:
    verify_checks(df, REFERENCE_CHECKS, "T8")

## TOPIC 9: Relation between columns

If a rule is broken -> `quarantine` that row. These rules compare two columns

**Two rules need a closer look**:
1. A half-filled coordinate pair (only one of `x_coordinate`/`y_coordinate`, or only one of `latidue`/`longitude`, is missing).
2. `primary_type` must match its `iucr` code in the CPD list of IUCR codes (`silver.ref_iucr`).

Step:
1. Set Up
2. Profile
3. Quarantine
4. ~~Transform~~
5. Verify

In [0]:
# 1. [SET UP]

PRIMARY_TYPE_ALIASES = {"CRIM SEXUAL ASSAULT": "CRIMINAL SEXUAL ASSAULT"}

for old, new in PRIMARY_TYPE_ALIASES.items():
    df = df.withColumn("primary_type", when(col("primary_type") == old, new).otherwise(col("primary_type")))

# (1) the reference for iucr -> primary_type: (table, column with the iucr code, column with its primary description)

IUCR_REFERENCE = (f"{CATALOG}.{SILVER_SCHEMA}.ref_iucr", "iucr", "primary_description")

# (2) load the allowed (iucr, primary_type) pairs and turn them into a rule.

iucr_pair_checks = []
ref_table, ref_iucr_column, ref_primary_column = IUCR_REFERENCE

if not spark.catalog.tableExists(ref_table):
    print(f"[SKIP]-[T9][SET UP]: iucr -> primary_type: reference table {ref_table} does not exists yet")
else:
    ref_pairs = (
        spark.table(ref_table)
        .select(
            upper(trim(col(ref_iucr_column).cast("string"))).alias("iucr"),
            upper(trim(col(ref_primary_column).cast("string"))).alias("primary_type")
        )
        .filter(col("iucr").isNotNull() & col("primary_type").isNotNull())
        .distinct()
        .collect()
    )

    if not ref_pairs:
        print(f"[SKIP]-[T9][SET UP]: iucr -> primary_type: reference table {ref_table} is empty")
    else:
        known_iucr = sorted({r["iucr"] for r in ref_pairs})
        allowed_pairs = sorted({f"{r['iucr']}|{r['primary_type']}" for r in ref_pairs})
        
        iucr_pair_checks.append(
            make_check(
                "iucr_primary_type_mismatch",
                ["iucr", "primary_type"],
                coalesce(
                    col("iucr").isin(*known_iucr)
                    & col("primary_type").isNotNull()
                    & ~concat_ws("|", col("iucr"), col("primary_type")).isin(*allowed_pairs),
                    lit(False)
                ),
                "quarantine"
            )
        )

# (3) relations between columns

beat_number = col("beat").try_cast("int")
district_number = col("district").try_cast("int")

RELATION_CHECKS = [
    make_check(
        "beat_not_in_district", 
        ["beat", "district"], 
        coalesce(floor(beat_number/100) != district_number, lit(False)), 
        "quarantine"
    ),
    make_check(
        "year_differs_from_date", 
        ["year", "date"], 
        coalesce(year(col("date")) != col("year"), lit(False)), 
        "quarantine"
    ),
    make_check(
        "x_y_coordinate_incomplete", 
        ["x_coordinate", "y_coordinate"], 
        is_missing("x_coordinate") != is_missing("y_coordinate"), 
        "quarantine"
    ),
    make_check(
        "latitude_longitude_incomplete", 
        ["latitude", "longitude"], 
        is_missing("latitude") != is_missing("longitude"), 
        "quarantine"
    )
] + iucr_pair_checks

In [0]:
# 2. [PROFILE] 

# (1) rows that break a relation 

show_violations(df, RELATION_CHECKS, "T9")

# (2) iucr -> primary_type: which pairs do not match the reference, and how many rows each has?

if iucr_pair_checks:
    display(
        df.filter(counted_bad(iucr_pair_checks[0]))
        .groupBy("iucr", "primary_type").count()
        .orderBy(col("count").desc())
        .limit(30)
    )
else: 
    print("[SKIP]-[T9][PROFILE]: no iucr reference rule - nothing to show")

In [0]:
# 3. [QUARANTINE] mark the rows that break a relation 

df = mark_quarantine(df, RELATION_CHECKS, "relation")

In [0]:
# 4. [TRANSFORM] rules whose action is "null" (none for now)

df = apply_nulling(df, RELATION_CHECKS, "T9")

In [0]:
# 5. [VERIFY] every row that breaks a relation is marked (FAIL otherwise)

verify_checks(df, RELATION_CHECKS, "T9")

## TOPIC 10: Decompose `date` and `block`

quarantine when: a `block` that has the right pattern but whose last word is not known street type is marked.

Step:
1. Set Up
2. Profile
3. Quarantine
4. Transform
5. Verify

In [0]:
# 1. [SET UP] last words of a block that count as a street type: the official USPS street-suffix (https://pe.usps.com/text/pub28/28apc_002.htm)

STREET_TYPES = list(STREET_TYPE_NAMES)

# the pieces of one block, as expressions
# - block_norm: block with single spaces
# - street_rest: the part after the direction
# - words: street_rest split by spaces
# - last_word: the last of those words
# - has_type: there is more than one word AND the last word is a known street type

block_norm = regexp_replace(col("block"), r"\s+", " ")
matches_pattern = block_norm.rlike(BLOCK_PATTERN)
street_rest = when(matches_pattern, regexp_extract(block_norm, BLOCK_PATTERN, 3))
words = split(street_rest, " ")
last_word = element_at(words, -1)
has_type = coalesce((size(words) > 1) & last_word.isin(*STREET_TYPES), lit(False))

BLOCK_CHECKS = [
    # True when the block has the right pattern but no known street type
    make_check("street_type_not_recognised", ["block"], coalesce(matches_pattern & ~has_type, lit(False)), "quarantine")
]

In [0]:
# 2. [PROFILE] blocks whose last word is not a known street type 

show_violations(df, BLOCK_CHECKS, "T10")

In [0]:
# 3. [QUARANTINE] mark the rows whose block has no known street type

df = mark_quarantine(df, BLOCK_CHECKS, "block")

In [0]:
# 4. [TRANSFORM] 

# (1) date -> incident_date (date part) and incident_hour (0-23)

df = (
    df
    .withColumn("incident_date", to_date(col("date")))
    .withColumn("incident_hour", hour(col("date")))
)

# (2) block -> block_hundred and direction 

df = (
    df
    .withColumn("block_hundred", when(matches_pattern, regexp_extract(block_norm, BLOCK_PATTERN, 1)))
    .withColumn("direction" ,when(matches_pattern, regexp_extract(block_norm, BLOCK_PATTERN, 2)))
)

# (3) the rest of the address -> street_name and street_type.

df = (
    df
    .withColumn("street_type", when(has_type, last_word))
    .withColumn("street_name", when(has_type, array_join(slice(words, 1, size(words) - 1), " ")).otherwise(street_rest))
)

In [0]:
# 5. [VERIFY] every block without a known street type is marked, and the new columns exist (FAIL otherwise)

verify_checks(df, BLOCK_CHECKS, "T10")

new_columns = ["incident_date", "incident_hour", "block_hundred", "direction", "street_name", "street_type"]
absent = [c for c in new_columns if c not in df.columns]

if absent:
    raise ValueError(f"[ERROR]-[T10][VERIFY]: columns were not created: {absent}")

print("[OK]-[T10][VERIFY]: date and block were decomposed")

## TOPIC 11: Table-level checks (Rows, Volumn, Freshness)

the rows must add up. A batch that is unusually large, or loaded long ago, is only a WARN. this topic looks at the table as a whole: no row may disappear silently (rows in = rows kept + rows quanrantined + old versions dropped)

Step:
1. Set Up
2. Profile
3. ~~Quarantine~~
4. Transform
5. Verify

In [0]:
# 1. [SET UP] limits 
VOLUME_WARN_ROWS = 100_000 # an incident batch bigger than this is ususual 
FRESHNESS_MAX_DAYS = 7 # Bronze data loaded more than this many days ago -> WARN

In [0]:
# 2. [PROFILE] 

# (1) where did every Bronze row of this batch go?

final_rows = df.filter(UNFLAGGED).count()
quarantined_total = df.filter(~UNFLAGGED).count()

print(f"Bronze rows in this batch: {raw_rows:,}")
print(f"    - older versions dropped (T4): {dropped_versions:,}")
print(f"    - quarantined (all topics): {quarantined_total:,}")
print(f"= rows that stay in Silver: {final_rows:,}")
print(f"= rows that break a quarantine rule, by topic (a row can be counted in several topics): {quarantine_counts}")

# (2) how old is the batch, and how recent are the incidents?

batch_age_days = (
    spark.createDataFrame([(batch_end,)], "batch_end timestamp")
    .select(datediff(current_timestamp(), col("batch_end")))
    .collect()[0][0]
)
newest_incident = unflagged(df).agg(max("date")).collect()[0][0]

print(f"[INFO]-[T11][PROFILE]: the newest Bronze row was loaded at {batch_end} ({batch_age_days} days ago)")
print(f"[INFO]-[T11][PROFILE]: the newest incident is {newest_incident}")

In [0]:
# 4. [TRANSFORM] the final Silver DataFrame: only the rows no rule marked; drop the copy of the original row and the reasons; add the processing time

df_silver = df.filter(UNFLAGGED).drop("_raw", "_reasons").withColumn("_processed_at", current_timestamp())
df_silver.printSchema()

In [0]:
# 5. [VERIFY] rows must add up (FAIL), volume and freshness (WARN)

accounted_for = quarantined_total + dropped_versions + final_rows

if accounted_for != raw_rows:
    raise ValueError(f"[ERROR]-[T11][VERIFY]: rows do not add up: {raw_rows:,} in, but {accounted_for:,} accounted for ({quarantined_total:,} quarantined + {dropped_versions:,} dropped + {final_rows:,} kept)")

print("[OK]-[T11][VERIFY]: every Bronze row is accounted for")

if last_watermark is not None and raw_rows > VOLUME_WARN_ROWS:
    print(f"[WARN]-[T11][VERIFY]: an incremental batch of {raw_rows:,} rows is bigger than {VOLUME_WARN_ROWS:,}")

if batch_age_days > FRESHNESS_MAX_DAYS:
    print(f"[WARN]-[T11][VERIFY]: the newest Bronze load is {batch_age_days} days old (limit {FRESHNESS_MAX_DAYS})")

# Write Quarantine Table

every marked row is stored once with its original Bronze values (all text, exactly as ingested in `_raw`) plus 
1. `quarantine_reason`: all the rules the row broke
2. `quarantine_at`: when this notebook quarantined it

In [0]:
# the marked rows with the original Bronze columns + all their reasons in one text

df_quarantine = (
    df.filter(~UNFLAGGED)
    .select("_raw.*", concat_ws("; ", col("_reasons")).alias("quarantine_reason"))
    .withColumn("quarantined_at", current_timestamp())
)

In [0]:
# the SQL condition that describes "the Bronze rows processed in this run" - the same window as in part C

window_condition = f"_ingested_at <= TIMESTAMP '{batch_end}'"

if last_watermark is not None:
    window_condition += f" AND _ingested_at > TIMESTAMP '{last_watermark}'"

print(f"[INFO]-[QUARANTINE]: replace the quarantine rows where {window_condition}")

In [0]:
# replace the quarantine rows of this window. written even when there are 0 rows, so a row that was quarantined by an earlier run (and has since been fixed) dissappears from the table 

(
    df_quarantine
    .write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", window_condition)
    .saveAsTable(QUARANTINE_TABLE)
)

In [0]:
# summary of the quarantine of this run, by reason

quarantined_rows = df_quarantine.count()
quarantine_pct = 100.0 * quarantined_rows / raw_rows if raw_rows > 0 else 0.0

print(f"[INFO]-[QUARANTINE]: {quarantined_rows:,} rows quarantined = {quarantine_pct:.3f}% of {raw_rows:,} Bronze rows")

if quarantined_rows > 0:
    display(df_quarantine.groupBy("quarantine_reason").count().orderBy(col("count").desc()))

In [0]:
# FAIL: at most MAX_QUARANTINE_PCT % of the batch may be quarantined. This checks comes AFTER the write on purpose: when it fails you can still open the quarantine table and see which rows caused it.

if quarantine_pct > MAX_QUARANTINE_PCT:
    raise ValueError(f"QUARANTINE LIMIT EXCEEDED: {quarantine_pct:.3f}% of the batch was quarantined (limit {MAX_QUARANTINE_PCT}%). The source probably changed - check {QUARANTINE_TABLE}.")

print(f"[OK]-[QUARANTINE]: {quarantine_pct:.3f}% quarantined - within the {MAX_QUARANTINE_PCT}% limit")

# Write Silver Table

In [0]:
# one row per `id`

distinct_ids = df_silver.select(count_distinct("id")).collect()[0][0]

if distinct_ids != final_rows:
    raise ValueError(f"[ERROR]-[WRITE]: {final_rows:,} rows to write but only {distinct_ids:,} distinct ids")

print(f"[OK]-[WRITE]: {final_rows:,} rows, exactly one per id")

In [0]:
# first run: the table does not exist yet, so it is created from this batch

if not spark.catalog.tableExists(TARGET_TABLE):
    df_silver.write.format("delta").saveAsTable(TARGET_TABLE)
    inserted, updated = final_rows, 0
    print(f"[INFO]-[WRITE]: {TARGET_TABLE} did not exist - created it with {final_rows:,} rows")
else:
    df_silver.createOrReplaceTempView("silver_batch")

    merge_sql = f"""
        MERGE INTO {TARGET_TABLE} AS t
        USING silver_batch AS s
        ON t.id = s.id
        WHEN MATCHED AND s.updated_on >= t.updated_on THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """

    result = spark.sql(merge_sql).collect()[0]
    inserted, updated = result["num_inserted_rows"], result["num_updated_rows"]

print(f"[INFO]-[WRITE]: {inserted:,} inserted, {updated:,} updated")

# Move the Watermark

the watermark (`last_ingeseted_at` in `load_control`) moves to the end of this batch

In [0]:
# create the control table on the first run

if not spark.catalog.tableExists(CONTROL_TABLE):
    spark.createDataFrame([], CONTROL_SCHEMA).write.format("delta").saveAsTable(CONTROL_TABLE)
    print(f"[INFO]-[WATERMARK]: {CONTROL_TABLE} did not exist - created it")

In [0]:
# one row for `crimes`: insert it, or move its watermark forward

(
    spark.createDataFrame([("crimes", batch_end)], "table_name string, last_ingested_at timestamp")
    .withColumn("updated_at", current_timestamp())
    .createOrReplaceTempView("control_update")
)

control_sql = f"""
    MERGE INTO {CONTROL_TABLE} AS t
    USING control_update AS s
    ON t.table_name = s.table_name
    WHEN MATCHED AND s.last_ingested_at >= t.last_ingested_at THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
"""

spark.sql(control_sql)

In [0]:
# read the watermark back: it must be the end of this batch (FAIL otherwise)

saved_watermark = get_last_watermark()

if saved_watermark != batch_end:
    raise ValueError(f"[ERROR]-[WATERMARK]: expected {batch_end} in {CONTROL_TABLE}, found {saved_watermark}")

print(f"[OK]-[WATERMARK][VERIFY]: {CONTROL_TABLE} says crimes is loaded up to {saved_watermark}")